# EQViT → PyTorch: reproducible inference
This notebook mirrors the original EQViT two-stage concept: normalized 60-s 3-C waveform → P probability; amplitude-preserving 30-s P-centered waveform → magnitude. It also validates the shape/parameter contract before loading an external Torch checkpoint.

In [ ]:
from pathlib import Path
import os, numpy as np, torch, matplotlib.pyplot as plt
from eqvit_torch.models import EQViTPicker, EQViTMagnitude
DEVICE='cuda' if torch.cuda.is_available() else ('mps' if hasattr(torch.backends,'mps') and torch.backends.mps.is_available() else 'cpu')
print('device:', DEVICE)


In [ ]:
picker=EQViTPicker().to(DEVICE); mag=EQViTMagnitude().to(DEVICE)
print('picker parameters:',sum(p.numel() for p in picker.parameters()))
print('magnitude parameters:',sum(p.numel() for p in mag.parameters()))
with torch.no_grad():
 print('picker output',picker(torch.randn(1,6000,3,device=DEVICE)).shape)
 print('magnitude output',mag(torch.randn(1,3000,3,device=DEVICE)).shape)

## Load the existing Torch-format model
Set the path to the downloaded Figshare file. The loader deliberately reports the serialization type; do not assume a state-dict naming scheme.

In [ ]:
from eqvit_torch.weights import load_legacy_torch_model
MODEL_PATH=Path('../weights/model_torch.pt')
if MODEL_PATH.exists():
 obj=load_legacy_torch_model(MODEL_PATH,DEVICE); print(type(obj)); print(list(obj)[:20] if isinstance(obj,dict) else obj)
else: print('Place Figshare file 31189627 at',MODEL_PATH)

In [ ]:
# Visual smoke test with synthetic P onset
t=np.arange(6000)/100; x=0.02*np.random.default_rng(7).normal(size=(6000,3)).astype('float32'); x[2200:,0]+=0.1*np.sin(2*np.pi*7*t[:3800])
fig,ax=plt.subplots(figsize=(12,4)); ax.plot(t,x[:,0]); ax.axvline(22,ls='--',label='synthetic P'); ax.set(xlabel='Time (s)',ylabel='Amplitude',title='Inference input sanity check'); ax.legend(); plt.show()